# Structured output: getting valid JSON back

A job board turns recruiters' free-text posts into cards (role, city, salary range, remote rule) with a small
hosted model. In this notebook you check the model's eight real replies in code: first with a plain Python
checker, then with Pydantic. Then you replay the retry loop from the lesson and see why a missing fact must be
allowed as "unknown".

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no account and no key.
The model's answers were recorded on 2026-11-13 (model `google/gemini-2.5-flash-lite` through OpenRouter), and the
notebook replays them, so you get the same answers the lesson showed.

## 1. Install Pydantic

The next cell installs the Pydantic version the lesson ran (2.14.3), so the error messages match the lesson word for word.

**Look for:** the line `pydantic 2.14.3`. pip may print a few lines about other packages; you can ignore them.

In [1]:
%pip install -q pydantic==2.14.3
import pydantic
print("pydantic", pydantic.VERSION)

Note: you may need to restart the kernel to use updated packages.
pydantic 2.14.3


## 2. The prompt, the eight posts and the recorded replies

`SYSTEM` is the instruction sent with every post. `POSTS` holds the eight recruiter posts. `REPLIES` holds what
the model answered for each one, copied exactly from the recordings.

**Look for:** 8 posts and 8 replies. The reply to post 1 starts with a ```` ```json ```` fence line, not with `{`.

In [2]:
SYSTEM = "You turn a recruiter's job post into a job card for our job board. Reply with JSON only, with these keys: role, city, salary_min, salary_max, remote. salary_min and salary_max are yearly amounts in euros, as numbers. remote is one of: onsite, hybrid, remote."

POSTS = {
    "post-1": "We're hiring a Backend Engineer (Python) for our Lisbon office. Pay: €55,000 to €70,000 a year. Hybrid: two days a week in the office.",
    "post-2": "Data Analyst wanted in Rotterdam! 45-52k depending on experience. Fully on-site, our team sits together.",
    "post-3": "Senior React developer, 100% remote, anywhere in the EU. Salary €80k–95k.",
    "post-4": "Join us as a QA Tester in Porto. Competitive salary and great benefits. Office-based.",
    "post-5": "Junior DevOps Engineer, Dublin. €42,000 - €50,000. Mostly remote, but we meet in the office once a month.",
    "post-6": "Product Designer for our Berlin or Munich studio. €4,500 per month. Hybrid.",
    "post-7": "Mobile developer (Flutter), Valencia. Up to €60,000. Remote-first, office optional.",
    "post-8": "Machine Learning Engineer in Tallinn. Salary from 70 000 EUR, upper limit open for the right person. Work from home on Fridays.",
}

# Recorded model replies (2026-11-13, google/gemini-2.5-flash-lite, temperature 0), copied exactly.
REPLIES = {
    "post-1": "```json\n{\n  \"role\": \"Backend Engineer (Python)\",\n  \"city\": \"Lisbon\",\n  \"salary_min\": 55000,\n  \"salary_max\": 70000,\n  \"remote\": \"hybrid\"\n}\n```",
    "post-2": "```json\n{\n  \"role\": \"Data Analyst\",\n  \"city\": \"Rotterdam\",\n  \"salary_min\": 45000,\n  \"salary_max\": 52000,\n  \"remote\": \"onsite\"\n}\n```",
    "post-3": "```json\n{\n  \"role\": \"Senior React developer\",\n  \"city\": \"Anywhere in the EU\",\n  \"salary_min\": 80000,\n  \"salary_max\": 95000,\n  \"remote\": \"remote\"\n}\n```",
    "post-4": "```json\n{\n  \"role\": \"QA Tester\",\n  \"city\": \"Porto\",\n  \"salary_min\": null,\n  \"salary_max\": null,\n  \"remote\": \"onsite\"\n}\n```",
    "post-5": "```json\n{\n  \"role\": \"Junior DevOps Engineer\",\n  \"city\": \"Dublin\",\n  \"salary_min\": 42000,\n  \"salary_max\": 50000,\n  \"remote\": \"hybrid\"\n}\n```",
    "post-6": "```json\n{\n  \"role\": \"Product Designer\",\n  \"city\": \"Berlin or Munich\",\n  \"salary_min\": 54000,\n  \"salary_max\": 54000,\n  \"remote\": \"hybrid\"\n}\n```",
    "post-7": "```json\n{\n  \"role\": \"Mobile developer (Flutter)\",\n  \"city\": \"Valencia\",\n  \"salary_min\": 0,\n  \"salary_max\": 60000,\n  \"remote\": \"remote\"\n}\n```",
    "post-8": "```json\n{\n  \"role\": \"Machine Learning Engineer\",\n  \"city\": \"Tallinn\",\n  \"salary_min\": 70000,\n  \"salary_max\": null,\n  \"remote\": \"hybrid\"\n}\n```",
}

print("posts:", len(POSTS), "| replies:", len(REPLIES))
print("reply to post-1, as received:")
print(REPLIES["post-1"])

posts: 8 | replies: 8
reply to post-1, as received:
```json
{
  "role": "Backend Engineer (Python)",
  "city": "Lisbon",
  "salary_min": 55000,
  "salary_max": 70000,
  "remote": "hybrid"
}
```


## 3. A checker in plain Python

`strip_fence` removes the ```` ``` ```` lines around a reply. `check_card` runs the checks in order and returns the
first problem it finds, or `"valid"`: every field is there, both salaries are whole numbers above 0, remote is an
allowed value, and salary_min is not above salary_max.

**Look for:** `valid` for 5 replies, and a salary message for posts 4, 7 and 8.

In [3]:
import json

def strip_fence(reply):
    lines = reply.strip().splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].startswith("```"):
        lines = lines[:-1]
    return "\n".join(lines)

REQUIRED = ["role", "city", "salary_min", "salary_max", "remote"]
ALLOWED = ["onsite", "hybrid", "remote"]

def check_card(text):
    card = json.loads(text)
    for key in REQUIRED:
        if key not in card:
            return f"{key}: missing"
    for key in ["salary_min", "salary_max"]:
        if type(card[key]) is not int or card[key] <= 0:
            return f"{key}: not a whole number above 0"
    if card["remote"] not in ALLOWED:
        return "remote: not an allowed value"
    if card["salary_min"] > card["salary_max"]:
        return "salary_min: above salary_max"
    return "valid"

for name, reply in REPLIES.items():
    print(name, "->", check_card(strip_fence(reply)))

post-1 -> valid
post-2 -> valid
post-3 -> valid
post-4 -> salary_min: not a whole number above 0
post-5 -> valid
post-6 -> valid
post-7 -> salary_min: not a whole number above 0
post-8 -> salary_max: not a whole number above 0


## 4. Broken on purpose: forget to remove the fence

This cell gives `check_card` the raw reply to post 1, with the ```` ```json ```` line still on top. It fails on purpose;
the `try` catches the error and prints it. Then two test inputs we wrote (not model replies): a salary given as
text, and a minimum above the maximum.

**Look for:** `JSONDecodeError: Expecting value: line 1 column 1 (char 0)`. The fields are fine; `json.loads` stops at
the very first character. Then the two test inputs are stopped by two different checks.

In [4]:
try:
    print(check_card(REPLIES["post-1"]))
except json.JSONDecodeError as err:
    print("JSONDecodeError:", err)

TEST_STRSAL = '{"role": "Data Analyst", "city": "Rotterdam", "salary_min": "45000", "salary_max": 52000, "remote": "onsite"}'
TEST_MINMAX = '{"role": "Data Analyst", "city": "Rotterdam", "salary_min": 52000, "salary_max": 45000, "remote": "onsite"}'
print("string salary ->", check_card(TEST_STRSAL))
print("min above max ->", check_card(TEST_MINMAX))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)
string salary -> salary_min: not a whole number above 0
min above max -> salary_min: above salary_max


## 5. The same rules with Pydantic

`JobCard` writes the rules as a class. `Field(gt=0)` means greater than 0, `Literal` lists the allowed values, and
the validator compares the two salaries. `JobCard.model_validate_json(text)` reads the JSON and checks it in one call.
It returns a card or raises a `ValidationError` that lists **every** error, not only the first.

**Look for:** the same 5 valid cards. Post 4 has 2 errors (`int_type` on both salaries), post 7 has 1 (`greater_than`,
input 0), post 8 has 1 (`int_type` on salary_max, input None).

In [5]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError, model_validator, ConfigDict

class JobCard(BaseModel):
    role: str
    city: str
    salary_min: int = Field(gt=0)
    salary_max: int = Field(gt=0)
    remote: Literal["onsite", "hybrid", "remote"]

    @model_validator(mode="after")
    def min_not_above_max(self):
        if self.salary_min > self.salary_max:
            raise ValueError("salary_min is above salary_max")
        return self

def validate_all(model):
    for name, reply in REPLIES.items():
        try:
            card = model.model_validate_json(strip_fence(reply))
            print(name, "-> valid:", card.model_dump())
        except ValidationError as err:
            print(name, "-> ValidationError, error_count =", err.error_count())
            for e in err.errors():
                print("   loc:", e["loc"], "| type:", e["type"], "| msg:", e["msg"], "| input:", repr(e["input"]))

validate_all(JobCard)

post-1 -> valid: {'role': 'Backend Engineer (Python)', 'city': 'Lisbon', 'salary_min': 55000, 'salary_max': 70000, 'remote': 'hybrid'}
post-2 -> valid: {'role': 'Data Analyst', 'city': 'Rotterdam', 'salary_min': 45000, 'salary_max': 52000, 'remote': 'onsite'}
post-3 -> valid: {'role': 'Senior React developer', 'city': 'Anywhere in the EU', 'salary_min': 80000, 'salary_max': 95000, 'remote': 'remote'}
post-4 -> ValidationError, error_count = 2
   loc: ('salary_min',) | type: int_type | msg: Input should be a valid integer | input: None
   loc: ('salary_max',) | type: int_type | msg: Input should be a valid integer | input: None
post-5 -> valid: {'role': 'Junior DevOps Engineer', 'city': 'Dublin', 'salary_min': 42000, 'salary_max': 50000, 'remote': 'hybrid'}
post-6 -> valid: {'role': 'Product Designer', 'city': 'Berlin or Munich', 'salary_min': 54000, 'salary_max': 54000, 'remote': 'hybrid'}
post-7 -> ValidationError, error_count = 1
   loc: ('salary_min',) | type: greater_than | msg: In

## 6. Two printed errors

First the full message Pydantic prints for post 4. It names each field and the problem; later this kind of text is
fed back to the model. Then lax and strict: by default Pydantic converts a value when it safely can. `StrictJobCard`
is the same class with strict mode on. Both get the test input where salary_min is the text `"45000"`.

**Look for:** `2 validation errors for JobCard`, then `salary_min` and `salary_max`, each with
`input_value=None, input_type=NoneType`. Then the lax line shows `'salary_min': 45000` as a number, and the strict
line is a `ValidationError` with `input_value='45000', input_type=str`.

In [6]:
try:
    JobCard.model_validate_json(strip_fence(REPLIES["post-4"]))
except ValidationError as err:
    print(err)
print()

class StrictJobCard(JobCard):
    model_config = ConfigDict(strict=True)

print("lax   :", JobCard.model_validate_json(TEST_STRSAL).model_dump())
try:
    StrictJobCard.model_validate_json(TEST_STRSAL)
except ValidationError as err:
    print("strict:", err)

2 validation errors for JobCard
salary_min
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/int_type
salary_max
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.14/v/int_type

lax   : {'role': 'Data Analyst', 'city': 'Rotterdam', 'salary_min': 45000, 'salary_max': 52000, 'remote': 'onsite'}
strict: 1 validation error for StrictJobCard
salary_min
  Input should be a valid integer [type=int_type, input_value='45000', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/int_type


## 7. A replayed model for the retry loop

The retry loop sends post 8 again. You have no API key, so `ask_model` does not call an API: it looks up the
recorded answer whose messages match the request exactly, and replays it. There are four recordings: post 8, the same
request sent twice more, and the request with the error fed back. A request with no recording raises `LookupError`.

**Look for:** 4 recordings. The first three were sent with exactly the same messages.

In [7]:
FIRST = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": POSTS["post-8"]},
]
# The message the fixed loop sent back to the model (recorded request, copied exactly).
FEEDBACK_SENT = "Your reply failed validation:\n1 validation error for JobCard\nsalary_max\n  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]\n    For further information visit https://errors.pydantic.dev/2.14/v/int_type\nReply again with the corrected JSON only."

# Recorded answers (2026-11-13), copied exactly: (name, the messages sent, the reply).
RECORDED = [
    ("post-8", FIRST, REPLIES["post-8"]),
    ("retry-same-2", FIRST, "```json\n{\n  \"role\": \"Machine Learning Engineer\",\n  \"city\": \"Tallinn\",\n  \"salary_min\": 70000,\n  \"salary_max\": null,\n  \"remote\": \"hybrid\"\n}\n```"),
    ("retry-same-3", FIRST, "```json\n{\n  \"role\": \"Machine Learning Engineer\",\n  \"city\": \"Tallinn\",\n  \"salary_min\": 70000,\n  \"salary_max\": null,\n  \"remote\": \"hybrid\"\n}\n```"),
    ("feedback-1", FIRST + [{"role": "assistant", "content": REPLIES["post-8"]},
                            {"role": "user", "content": FEEDBACK_SENT}],
     "```json\n{\n  \"role\": \"Machine Learning Engineer\",\n  \"city\": \"Tallinn\",\n  \"salary_min\": 70000,\n  \"salary_max\": 1000000,\n  \"remote\": \"hybrid\"\n}\n```"),
]

def without_help_link(messages):
    # Pydantic's help link names its version. Drop that line, so another Pydantic version still matches.
    return [{**m, "content": "\n".join(l for l in m["content"].split("\n") if "For further information" not in l)}
            for m in messages]

used = set()
def ask_model(messages):
    for name, sent, reply in RECORDED:
        if name not in used and without_help_link(sent) == without_help_link(messages):
            used.add(name)
            print(f"   (replaying recording {name})")
            return reply
    raise LookupError("no recording for these messages")

print("recordings:", len(RECORDED))
print("first three sent the same messages:", all(sent == FIRST for _, sent, _ in RECORDED[:3]))

recordings: 4
first three sent the same messages: True


## 8. Broken on purpose: a retry loop that never changes the request

The team's first retry: if the reply fails validation, ask again, up to three times. The error is only printed.

**Look for:** three attempts, each with `salary_max: Input should be a valid integer, input None`, then
`gave up after 3 attempts` and `result: None`. Same messages, temperature 0, same reply.

In [8]:
def import_post_buggy(messages):
    for attempt in range(3):
        reply = ask_model(messages)
        try:
            return JobCard.model_validate_json(strip_fence(reply))
        except ValidationError as err:
            e = err.errors()[0]
            print(f"attempt {attempt}: {e['loc'][0]}: {e['msg']}, input {e['input']!r}")
    print("gave up after 3 attempts")

used.clear()
print("result:", import_post_buggy(list(FIRST)))

   (replaying recording post-8)
attempt 0: salary_max: Input should be a valid integer, input None
   (replaying recording retry-same-2)
attempt 1: salary_max: Input should be a valid integer, input None
   (replaying recording retry-same-3)
attempt 2: salary_max: Input should be a valid integer, input None
gave up after 3 attempts
result: None


## 9. Feed the error back

The fixed loop adds the model's reply and Pydantic's message to the conversation before the next attempt. The cell
also prints the message it sent, so you can compare it with the recorded one.

**Look for:** one failed attempt, then a valid card with `'salary_max': 1000000`. The post says the upper limit is
open, so this number is made up. It passes validation anyway: the shape is fixed, the truth is not.

In [9]:
def import_post_fixed(messages):
    for attempt in range(3):
        reply = ask_model(messages)
        try:
            return JobCard.model_validate_json(strip_fence(reply))
        except ValidationError as err:
            e = err.errors()[0]
            print(f"attempt {attempt}: {e['loc'][0]}: {e['msg']}, input {e['input']!r}")
            messages = messages + [
                {"role": "assistant", "content": reply},
                {"role": "user", "content": "Your reply failed validation:\n" + str(err) + "\nReply again with the corrected JSON only."}]
    print("gave up after 3 attempts")

used.clear()
card = import_post_fixed(list(FIRST))
print("result:", card.model_dump() if card else None)

   (replaying recording post-8)
attempt 0: salary_max: Input should be a valid integer, input None
   (replaying recording feedback-1)
result: {'role': 'Machine Learning Engineer', 'city': 'Tallinn', 'salary_min': 70000, 'salary_max': 1000000, 'remote': 'hybrid'}


## 10. Let the schema say "unknown"

The real fix: a post with no salary is not an error. In `JobCardV2` both salaries may be `None`. The min and max
check runs only when both are numbers.

**Look for:** posts 4 and 8 are now valid, with `None` where the post gives no amount. Post 7 still fails on
salary_min = 0: the model wrote 0 where null belonged.

In [10]:
class JobCardV2(BaseModel):
    role: str
    city: str
    salary_min: int | None = Field(default=None, gt=0)
    salary_max: int | None = Field(default=None, gt=0)
    remote: Literal["onsite", "hybrid", "remote"]

    @model_validator(mode="after")
    def min_not_above_max(self):
        if self.salary_min is not None and self.salary_max is not None and self.salary_min > self.salary_max:
            raise ValueError("salary_min is above salary_max")
        return self

validate_all(JobCardV2)

post-1 -> valid: {'role': 'Backend Engineer (Python)', 'city': 'Lisbon', 'salary_min': 55000, 'salary_max': 70000, 'remote': 'hybrid'}
post-2 -> valid: {'role': 'Data Analyst', 'city': 'Rotterdam', 'salary_min': 45000, 'salary_max': 52000, 'remote': 'onsite'}
post-3 -> valid: {'role': 'Senior React developer', 'city': 'Anywhere in the EU', 'salary_min': 80000, 'salary_max': 95000, 'remote': 'remote'}
post-4 -> valid: {'role': 'QA Tester', 'city': 'Porto', 'salary_min': None, 'salary_max': None, 'remote': 'onsite'}
post-5 -> valid: {'role': 'Junior DevOps Engineer', 'city': 'Dublin', 'salary_min': 42000, 'salary_max': 50000, 'remote': 'hybrid'}
post-6 -> valid: {'role': 'Product Designer', 'city': 'Berlin or Munich', 'salary_min': 54000, 'salary_max': 54000, 'remote': 'hybrid'}
post-7 -> ValidationError, error_count = 1
   loc: ('salary_min',) | type: greater_than | msg: Input should be greater than 0 | input: 0
post-8 -> valid: {'role': 'Machine Learning Engineer', 'city': 'Tallinn', '

## 11. The provider's schema mode

Post 8 was also sent with `response_format` type `json_schema`, strict on, twice: once with salary_max required as an
integer, once with salary_max allowed to be an integer or null. The two recorded replies are below, each checked
with `JobCardV2`. `JobCard.model_json_schema()` shows the schema Pydantic would write for you.

**Look for:** neither reply has a fence. With an integer forced, salary_max is `100000`, an invented number. With null
allowed, it is `None`. Both are valid cards, so validation alone cannot tell you which one is true.

In [11]:
SCHEMA_REPLIES = {
    "salary_max must be an integer": "{\n\"role\": \"Machine Learning Engineer\",\n\"city\": \"Tallinn\",\n\"salary_min\": 70000,\n\"salary_max\": 100000,\n\"remote\": \"hybrid\"\n}",
    "salary_max may be an integer or null": "{\n\"role\": \"Machine Learning Engineer\",\n\"city\": \"Tallinn\",\n\"salary_min\": 70000,\n\"salary_max\": null,\n\"remote\": \"hybrid\"\n}",
}
for label, reply in SCHEMA_REPLIES.items():
    print(label)
    print("   starts with a fence:", reply.startswith("```"))
    print("   JobCardV2:", JobCardV2.model_validate_json(reply).model_dump())

print()
print("JobCard.model_json_schema() required:", JobCard.model_json_schema()["required"])
print("salary_max:", JobCard.model_json_schema()["properties"]["salary_max"])

salary_max must be an integer
   starts with a fence: False
   JobCardV2: {'role': 'Machine Learning Engineer', 'city': 'Tallinn', 'salary_min': 70000, 'salary_max': 100000, 'remote': 'hybrid'}
salary_max may be an integer or null
   starts with a fence: False
   JobCardV2: {'role': 'Machine Learning Engineer', 'city': 'Tallinn', 'salary_min': 70000, 'salary_max': None, 'remote': 'hybrid'}

JobCard.model_json_schema() required: ['role', 'city', 'salary_min', 'salary_max', 'remote']
salary_max: {'exclusiveMinimum': 0, 'title': 'Salary Max', 'type': 'integer'}


## Now change it

Three small experiments. In each cell, edit the line marked `# <- change this`, run the cell, and compare.

**Experiment 1: text Pydantic cannot convert.** Change `"45000"` to `"45k"`. Lax mode converted `"45000"` to a number.
`"45k"` is not a number at all, so now the lax `JobCard` should fail too, with type `int_parsing`.

In [12]:
salary_min = '"45000"'  # <- change this to '"45k"'
text = TEST_STRSAL.replace('"45000"', salary_min)
try:
    print("lax:", JobCard.model_validate_json(text).model_dump())
except ValidationError as err:
    print(err)

lax: {'role': 'Data Analyst', 'city': 'Rotterdam', 'salary_min': 45000, 'salary_max': 52000, 'remote': 'onsite'}


**Experiment 2: the rule that compares two fields.** Change salary_min from `45000` to `60000`, above salary_max (52000).
Each field alone is still fine, so only the validator can catch it: you should see one `value_error` saying
`salary_min is above salary_max`.

In [13]:
card = {"role": "Data Analyst", "city": "Rotterdam", "salary_min": 45000, "salary_max": 52000, "remote": "onsite"}
card["salary_min"] = 45000  # <- change this to 60000
try:
    print("valid:", JobCardV2.model_validate(card).model_dump())
except ValidationError as err:
    print(err)

valid: {'role': 'Data Analyst', 'city': 'Rotterdam', 'salary_min': 45000, 'salary_max': 52000, 'remote': 'onsite'}


**Experiment 3: a fix that hides the problem.** Change `gt=0` to `ge=0` (greater than or equal to 0) on salary_min.
Post 7 should now pass with salary_min 0. Every reply is valid, but the card says the job can pay 0 euros: the post
gave no lowest amount. Loosening a rule until the error goes away does not make the card true; null was the honest answer.

In [14]:
class JobCardLoose(JobCardV2):
    salary_min: int | None = Field(default=None, gt=0)  # <- change this: gt=0 to ge=0

validate_all(JobCardLoose)

post-1 -> valid: {'role': 'Backend Engineer (Python)', 'city': 'Lisbon', 'salary_min': 55000, 'salary_max': 70000, 'remote': 'hybrid'}
post-2 -> valid: {'role': 'Data Analyst', 'city': 'Rotterdam', 'salary_min': 45000, 'salary_max': 52000, 'remote': 'onsite'}
post-3 -> valid: {'role': 'Senior React developer', 'city': 'Anywhere in the EU', 'salary_min': 80000, 'salary_max': 95000, 'remote': 'remote'}
post-4 -> valid: {'role': 'QA Tester', 'city': 'Porto', 'salary_min': None, 'salary_max': None, 'remote': 'onsite'}
post-5 -> valid: {'role': 'Junior DevOps Engineer', 'city': 'Dublin', 'salary_min': 42000, 'salary_max': 50000, 'remote': 'hybrid'}
post-6 -> valid: {'role': 'Product Designer', 'city': 'Berlin or Munich', 'salary_min': 54000, 'salary_max': 54000, 'remote': 'hybrid'}
post-7 -> ValidationError, error_count = 1
   loc: ('salary_min',) | type: greater_than | msg: Input should be greater than 0 | input: 0
post-8 -> valid: {'role': 'Machine Learning Engineer', 'city': 'Tallinn', '

## What you should see

- Section 3: the plain checker prints `valid` for 5 of the 8 replies, and a salary message for posts 4, 7 and 8.
- Section 4: without `strip_fence`, `json.loads` raises `JSONDecodeError: Expecting value: line 1 column 1 (char 0)`.
- Section 6: Pydantic lists 2 errors for post 4, one for salary_min and one for salary_max, both of type `int_type`.
- Section 6: the text "45000" passes `JobCard` and fails `StrictJobCard`.
- Sections 8 and 9: the loop that resends the same messages gives up after 3 attempts; the loop that feeds the error back gets salary_max 1000000, a number the post never states.
- Section 10: with both salaries allowed to be None, posts 4 and 8 pass and post 7 still fails on salary_min = 0.